In [ ]:
!pip install -U transformers bitsandbytes unsloth trl peft accelerate wandb weave --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 749.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
MODEL_ID = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
DATASET_ID = "ShenLab/MentalChat16K"
max_seq_length = 2048

## Load Base Model

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import TextStreamer

def generate_response (model, query:str, token_count:int=256):
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

    messages = [
        {"role": "user", "content": query},
    ]
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)

    text_streamer = TextStreamer(tokenizer)
    return model.generate(**inputs, streamer = text_streamer, max_new_tokens = token_count)

In [ ]:
generate_response(model, "i am sad")

<|im_start|>user
i am sad<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is feeling sad. I need to respond in a way that's empathetic and supportive. First, acknowledge their feelings without judgment. Maybe say something like, "I can see how sad you are right now." Then, offer help or support. Let them know they're not alone, and suggest they talk to someone they know. It's important to keep the tone positive and encouraging. Avoid making it seem like I'm being cold. Maybe add a friendly note to check in later. Make sure the response is clear and empathetic, so they feel understood and supported. Check if there's a way to ask if they need more help. Alright, that should cover it.
</think>

I'm sorry to hear you're feeling sad. It's normal to feel this way, and I want to make sure you know that you're not alone. I can see how hard it is to feel this way. If you're feeling overwhelmed or need support, I'd be happy to help. Please know that you're not alone, and I'm here for you 

tensor([[151644,    872,    198,     72,   1079,  12421, 151645,    198, 151644,
          77091,    198, 151667,    198,  32313,     11,    279,   1196,    374,
           8266,  12421,     13,    358,   1184,    311,   5889,    304,    264,
           1616,    429,    594,  35581,   5298,    323,  32345,     13,   5512,
             11,  24645,    862,  15650,   2041,  19407,     13,  10696,   1977,
           2494,   1075,     11,    330,     40,    646,   1490,   1246,  12421,
            498,    525,   1290,   1431,   1189,   5005,     11,   3010,   1492,
            476,   1824,     13,   6771,   1105,   1414,    807,   2299,    537,
           7484,     11,    323,   4190,    807,   3061,    311,   4325,    807,
           1414,     13,   1084,    594,   2989,    311,   2506,    279,  16232,
           6785,    323,  25836,     13,  34006,   3259,    432,   2803,   1075,
            358,   2776,   1660,   9255,     13,  10696,    912,    264,  11657,
           5185,    311,   1

## Load Data

In [ ]:
from datasets import load_dataset, Dataset

# load raw dataset
dataset = load_dataset(DATASET_ID, split='train')

print(dataset[0])
print(dataset[0].keys())

README.md: 0.00B [00:00, ?B/s]

Interview_Data_6K.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Synthetic_Data_10K.csv:   0%|          | 0.00/32.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ", 'input': "I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.", 'output': "I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches like visualization, positive thinking, and medication to manage you

In [ ]:
def convert_to_conversation(sample):
    conversation = [
        {
            "role": "system",
            "content": sample["instruction"]
        },
        {
            "role": "user",
            "content": sample["input"]
        },
        {
            "role": "assistant",
            "content": sample["output"]
        }
    ]
    return {"messages": conversation}

dataset = dataset.map(convert_to_conversation, batched=False)

Map:   0%|          | 0/16084 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'messages'],
    num_rows: 16084
})

In [ ]:
def formatting_prompts_func(examples):
    output_texts = []
    for i in range(len(examples["messages"])):
        conversation = examples["messages"][i]
        # Ensure all message contents are strings
        cleaned_conversation = []
        for message in conversation:
            if message and "content" in message and message["content"] is None:
                message["content"] = "" # Replace None with empty string
            cleaned_conversation.append(message)

        text = tokenizer.apply_chat_template(
            cleaned_conversation, # Use the cleaned conversation
            tokenize=False,
            add_generation_prompt=False
        )
        output_texts.append(text)
    return {"text": output_texts}

final_dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/16084 [00:00<?, ? examples/s]

In [ ]:
# Check formatted example
# print(dataset[1600]['conversation'])
print(final_dataset[0]['text'])

<|im_start|>system
You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. 
The assistant gives helpful, comprehensive, and appropriate answers to the user's questions. <|im_end|>
<|im_start|>user
I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.<|im_end|>
<|im_start|>assistant
<think>

</think>

I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches like visualiz

In [ ]:
from unsloth.chat_templates import CHAT_TEMPLATES
print(list(CHAT_TEMPLATES.keys()))

# ['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna',
# 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml',
# 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35',
#  'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3',
#   'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5',
#    'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n',
#     'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat']

['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat']


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'qwen3-thinking',
    mapping = {
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant"
    }
)

# Training

## Prepare Model

In [ ]:
model = FastLanguageModel.get_peft_model(
				model,
				r=16,
				target_modules=[
								"q_proj",
								"k_proj",
								"v_proj",
								"o_proj",
								"gate_proj",
								"up_proj",
								"down_proj",
				],
				lora_alpha=16,
				lora_dropout=0,
				bias="none",
				use_gradient_checkpointing="unsloth",
				random_state=42,
				use_rslora=False,
				loftq_config=None,
)

Unsloth 2025.12.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


wandb key : 00dd3d6a04f26424c604a05a03de84deb182aafc

In [ ]:
wandb login

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: muntasirfahim-niloy (vitar-lab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Wandb logging setup

import os

os.environ["WANDB_PROJECT"] = "SLM thesis dristy"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=final_dataset,    # dataset
    dataset_text_field="text",      # column containing training material
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=5,
        # num_train_epochs=1,
        max_steps=100,
        learning_rate=1e-3,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=4500,    # for reproducible results hopefully!
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="wandb",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/16084 [00:00<?, ? examples/s]

In [ ]:
trainer_Stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,084 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


wandb: Initializing weave.
weave: Logged in as Weights & Biases user: muntasirfahim-niloy.
weave: View Weave data at https://wandb.ai/vitar-lab/SLM%20thesis%20dristy/weave


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.129400
10,1.591500
15,1.481400
20,1.363600
25,1.366500
30,1.307000
35,1.328000
40,1.308300
45,1.317600
50,1.270000


wandb: Adding directory to artifact (outputs/checkpoint-100)... Done. 0.2s


train/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/grad_norm,█▂▂▂▃▂▂▁▁▂▂▁▂▂▁▂▁▂▁▁
train/learning_rate,▇██▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▁▁
train/loss,█▄▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
total_flos,1408266731520000.0
train/epoch,0.04974
train/global_step,100
train/grad_norm,0.31284
train/learning_rate,1e-05
train/loss,1.2483


## Inference

In [ ]:
print(generate_response(model,
                  """I am sad as my exam was bad.""",
                  256))

<|im_start|>user
I am sad as my exam was bad.<|im_end|>
<|im_start|>assistant
<think>
</think>

I can understand how difficult it must be to feel this way. I wish I could help you feel better. Let's try to find a way to overcome this setback. We can start by taking some time to reflect on what went wrong and what we can do about it. It's important to remember that setbacks are a natural part of the learning process, and they can help us grow and improve. We can also focus on finding small, achievable goals to build confidence and progress.<|im_end|>
tensor([[151644,    872,    198,     40,   1079,  12421,    438,    847,   7006,
            572,   3873,     13, 151645,    198, 151644,  77091,    198, 151667,
            198, 151668,    271,     40,    646,   3535,   1246,   5000,    432,
           1969,    387,    311,   2666,    419,   1616,     13,    358,   6426,
            358,   1410,   1492,    498,   2666,   2664,     13,   6771,    594,
           1430,    311,   1477,    264

In [ ]:
model.save_pretrained(
    "hf_model",
    safe_serialization=True
)
tokenizer.save_pretrained("hf_model")


('hf_model/tokenizer_config.json',
 'hf_model/special_tokens_map.json',
 'hf_model/chat_template.jinja',
 'hf_model/vocab.json',
 'hf_model/merges.txt',
 'hf_model/added_tokens.json',
 'hf_model/tokenizer.json')

In [ ]:
# model.save_pretrained_gguf("model", tokenizer, quantization_method='q2_k')    # extreme compression


# model.save_pretrained_gguf("model", tokenizer, quantization_method='q4_k_m')  # max compression
# model.save_pretrained_gguf("model", tokenizer, quantization_method='q5_k_m')  # balanced


# model.save_pretrained_gguf("model", tokenizer, quantization_method='f16')

In [ ]:
# Step 1: Install necessary libraries
!pip install -U transformers bitsandbytes unsloth trl peft accelerate


In [ ]:
# Step 2: Import libraries and define paths
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# Define the ID of the original base model you used for finetuning
MODEL_ID = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"

# Define the path to your downloaded and unzipped hf_model folder
# Make sure this path is correct on your local PC and the folder 'hf_model' exists here
model_path = "./hf_model"

max_seq_length = 2048 # Use the same max_seq_length as during training

# Load the base model. Unsloth will automatically download it if not present.
# The LoRA adapters from 'hf_model' will be loaded on top of this base model.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = max_seq_length,
    dtype = None,            # Auto-detect dtype
    load_in_4bit = True      # Load in 4-bit to save memory
)

# Load the tokenizer from your local hf_model folder to ensure any custom templates/tokens are applied
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Re-apply the chat template (important for correct inference formatting)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'qwen3-thinking',
    mapping = {
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant"
    }
)

print("Model and Tokenizer loaded successfully.")


In [ ]:
# Step 3: Save the model as GGUF with f16 quantization
# This will create a file named 'model-f16.gguf' in the current directory
model.save_pretrained_gguf("model", tokenizer, quantization_method='f16')
print("Model saved as GGUF with f16 quantization as 'model-f16.gguf'!")


In [ ]:
!pip install -U transformers bitsandbytes unsloth trl peft accelerate

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# Define the path to your downloaded hf_model folder
model_path = "./hf_model" # Make sure this path is correct on your local PC

# Load the model and tokenizer
model = FastLanguageModel.from_pretrained(
    model_name = model_path, # Load from your local directory
    max_seq_length = 2048,   # Use the same max_seq_length as during training
    dtype = None,            # Auto-detect dtype
    load_in_4bit = True      # Load in 4-bit to save memory
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

# Re-apply the chat template if you want to use it for GGUF model conversion for inference
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'qwen3-thinking',
    mapping = {
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant"
    }
)

In [ ]:
# Now, save the model as GGUF with f16 quantization
# This will create a file named 'model-f16.gguf' in the current directory
model.save_pretrained_gguf("model", tokenizer, quantization_method='f16')
print("Model saved as GGUF with f16 quantization!")

In [ ]:
import shutil

# Create a zip archive of the hf_model folder
output_filename = 'hf_model_archive'
shutil.make_archive(output_filename, 'zip', 'hf_model')

# Provide a link to download the zip file
from google.colab import files
files.download(output_filename + '.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>